In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from diffrax import odeint, Dopri5, ODETerm, SaveAt
from types import SimpleNamespace

Вид записи,Формула,Плюсы
Классический,dtdxi​​=−∂xi​∂E​,Понятно всем без исключения.
Компактный,x˙i​=−∂i​E,"Минимум символов, высокая плотность информации."
Векторный,x˙=−∇E,Самый быстрый способ записи (без индексов вообще).

In [ ]:
def form_layer(name, in_dim, out_dim, bias = True, w_scale = 0.1, b_scale = 0.1, G = None):
    w = np.random.randn(in_dim, out_dim, out_dim)
    w = w - np.transpose(w, (0, 2, 1))
    if bias:
        b = np.random.randn(out_dim, out_dim)
        b = b - b.T
        def v_dir(u, v, w, b):
            return jnp.einsum("j, jki, k -> i", u, w, v, optimize=True) + b @ v
    else:
        def v_dir(u, v, w):
            return jnp.einsum("j, jki, k -> i", u, w, v, optimize=True)
    
    v_dir = jax.jit(v_dir)

    jax.grad()

    output = SimpleNamespace(
        w = w,
        b = w,
        out_fun = out_fun,
    )
    return 

class control_system:
    def __init__(self, dimentions, in_sys, w_scale = 0.1, b_scale = 0.1, keep_lenght = True):
        self.dims = dimentions
        self.in_sys = in_sys

        self.w = []
        self.b = []

        for dc, dn in zip(self.dims, self.dims[1:]):
            w = np.random.randn(dc, dn, dn)
            if keep_lenght:
                w = w - np.transpose(w, (0, 2, 1))
            self.w.append(w*w_scale)

            b = np.random.randn(dn, dn)
            b = b - b.T
            self.b.append(b_scale*b)

        self.state = []

        for d in self.dims:
            self.state.append(np.random.randn(d))

        def ds_dt(t, flat_state):
            # Reshape the flat state back to the original structure
            state = []
            idx = 0
            for d in self.dims:
                state.append(flat_state[idx:idx + d])
                idx += d

            ds = [None for _ in self.dims]
            ds[0] = self.in_sys(t, state[0])
            for i, (u, w, b, v) in enumerate(zip(state, self.w, self.b, state[1:])):
                ds[i + 1] = np.einsum("j, jki, k -> i", u, w, v, optimize=True) + b @ v

            # Flatten the derivatives back into a 1D array
            return np.concatenate(ds)

        self.ds_dt = ds_dt

    def run(self, t_span=(0, 50), initials_in_sys=None, res=5000):
        if initials_in_sys is not None:
            self.state[0] = initials_in_sys

        # Flatten the initial state into a 1D array
        flat_state = np.concatenate([np.ravel(s) for s in self.state])

        t_eval = np.linspace(t_span[0], t_span[1], res)
        solution = solve_ivp(self.ds_dt, t_span, flat_state, t_eval=t_eval)

        # Reshape the solution back to the original structure
        reshaped_solution = []
        for i in range(len(solution.t)):
            idx = 0
            state_at_t = []
            for d in self.dims:
                state_at_t.append(solution.y[idx:idx + d, i])
                idx += d
            reshaped_solution.append(state_at_t)

        # Adjust reshaped_solution to ensure uniformity across layers
        reshaped_solution = [
            np.array([state_at_t[layer] for state_at_t in reshaped_solution])
            for layer in range(len(self.dims))
        ]

        return solution.t, reshaped_solution


def show_ev(evolution, time, title, random_projections = False):
    if not random_projections:
        x, y, z = [evolution[:, i] if i<evolution.shape[-1] else np.zeros_like(evolution[:, 0]) for i in range(3)]
    else:
        proj = evolution @ np.random.randn(evolution.shape[-1], 3)
        x, y, z = [proj[:, i] for i in range(3)]
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color=time, width=2), ))
    fig.update_layout(
        title = title,
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z'
        ),
        template='plotly_dark',
    )

    fig.show()